In [50]:
import numpy as np
from sympy import *

import torch
# from torch import nn
from torch.nn.parameter import Parameter
import torch.nn.functional as F
import torch.optim as optim
import torch.nn.init as init

%matplotlib inline
import matplotlib
import matplotlib.pyplot as plt

In [ ]:
torch.rand(10) * 0.5

In [17]:
# t, delta, omega = symbols("t delta omega")
t = Symbol("t", positive=True)

m = 1
mu = 5
k = 500

delta = mu/(2*m)
w0 = sqrt(k)
w = sqrt(w0**2 - delta**2)

phi = tan(-delta/w)**(-1)
A = 1/(2 * cos(phi))

display(A)

x = exp(-delta*t)*(2*A*cos(phi + w*t))
display(x.simplify())

-0.595470930409999

-1.19094186082*exp(-2.5*t)*cos(22.220486043289*t - 8.85065980104099)

In [35]:
n = 10

times_train = (np.random.rand(n) * 0.5).astype("float32")
D_train = (np.array([x.subs(t, time) for time in times_train])).astype("float32")
print(times_train)
print(D_train, "\n")


# create validation data
times_val = (np.random.rand(n) * 0.5).astype("float32")
D_val = (np.array([x.subs(t, time) for time in times_val])).astype("float32")
print(times_val)
print(D_val)

[0.2565484  0.47353992 0.37033448 0.27149448 0.3082405  0.31222937
 0.4422975  0.07040991 0.1248476  0.07147331]
[ 0.62709194  0.03669519 -0.3835772   0.5727451   0.23003878  0.18297355
 -0.22040442 -0.53714454 -0.8530895  -0.55540943] 

[0.1051741  0.1507489  0.01632371 0.35994673 0.04788768 0.07926321
 0.40360487 0.21241547 0.32350183 0.01301449]
[-0.8913805  -0.5795216   0.67723095 -0.31870785 -0.07116849 -0.67622083
 -0.43119153  0.38476053  0.04846714  0.74924797]


In [52]:
simple_model = torch.nn.Sequential(
    # input layer with 1 input
    torch.nn.Linear(1, 10),
    torch.nn.Tanh(),
    # hidden layer
    torch.nn.Linear(10, 1)
)

print(simple_model)
net = simple_model

Sequential(
  (0): Linear(in_features=1, out_features=10, bias=True)
  (1): Tanh()
  (2): Linear(in_features=10, out_features=1, bias=True)
)


In [53]:
# num_features = 1
# num_l1 = 10
# num_out = 1

# class Net(nn.Module):

#     def __init__(self, num_features, num_hidden, num_output):
#         super(Net, self).__init__()  

#         # self.do = nn.Dropout(0.1)
#         # self.bn = nn.BatchNorm1d(num_features)
#         # input layer
#         self.W_1 = Parameter(init.kaiming_normal_(torch.Tensor(num_hidden, num_features)))
#         self.b_1 = Parameter(init.constant_(torch.Tensor(num_hidden), 0))
#         # hidden layer
#         self.W_2 = Parameter(init.xavier_normal_(torch.Tensor(num_hidden, num_hidden)))
#         self.b_2 = Parameter(init.constant_(torch.Tensor(num_hidden), 0))
#         # define activation function in constructor
#         self.activation = torch.nn.Tanh()

#     def forward(self, x):
#         x = self.bn(x)
#         x = F.linear(x, self.W_1, self.b_1)
#         x = self.activation(x)
#         x = F.linear(x, self.W_2, self.b_2)
#         return x

# net = Net(num_features, num_l1, num_out)

In [54]:
optimizer = optim.SGD(net.parameters(), lr=0.01)
criterion = torch.nn.MSELoss()

In [56]:
loss_fn = torch.nn.MSELoss(reduction='sum')
model = simple_model

x_train = torch.from_numpy(times_train)
x_val = torch.from_numpy(times_val)
target_train = torch.from_numpy(D_train)
target_val = torch.from_numpy(D_val)

learning_rate = 1e-4
for t in range(500):
    # Forward pass: compute predicted y by passing x to the model. Module objects
    # override the __call__ operator so you can call them like functions. When
    # doing so you pass a Tensor of input data to the Module and it produces
    # a Tensor of output data.
    for x, y in zip(x_train, target_train):
        x = x.reshape(1,1)
        y = y.reshape(1,1)
        y_pred = model(x)

        # Compute and print loss. We pass Tensors containing the predicted and true
        # values of y, and the loss function returns a Tensor containing the
        # loss.
        loss = loss_fn(y_pred, y)
        print(t, loss.item())

        # Zero the gradients before running the backward pass.
        model.zero_grad()

        # Backward pass: compute gradient of the loss with respect to all the learnable
        # parameters of the model. Internally, the parameters of each Module are stored
        # in Tensors with requires_grad=True, so this call will compute gradients for
        # all learnable parameters in the model.
        loss.backward()

        # Update the weights using gradient descent. Each parameter is a Tensor, so
        # we can access its gradients like we did before.
        with torch.no_grad():
            for param in model.parameters():
                param -= learning_rate * param.grad

0 0.15396903455257416
0 0.017825912684202194
0 0.3425826132297516
0 0.11730404943227768
0 0.00010256202222080901
0 0.0012804769212380052
0 0.16002561151981354
0 0.6731396913528442
0 1.2596231698989868
0 0.7007070183753967
1 0.1557028889656067
1 0.017258424311876297
1 0.3400450348854065
1 0.11881262809038162
1 0.00015159293252509087
1 0.0011290452675893903
1 0.1583143025636673
1 0.6694921851158142
1 1.2546532154083252
1 0.6969912648200989
2 0.15743425488471985
2 0.016704026609659195
2 0.3375347852706909
2 0.12032020837068558
2 0.00020972592756152153
2 0.000988095416687429
2 0.15662410855293274
2 0.6658796072006226
2 1.2497273683547974
2 0.6933106780052185
3 0.15916304290294647
3 0.016162484884262085
3 0.3350510597229004
3 0.12182671576738358
3 0.00027677827165462077
3 0.0008574261446483433
3 0.15495480597019196
3 0.6623013615608215
3 1.2448450326919556
3 0.6896651387214661
4 0.16088922321796417
4 0.01563357748091221
4 0.3325938582420349
4 0.12333198636770248
4 0.00035254095564596355
4 0

In [ ]:
# we could have done this ourselves,
# but we should be aware of sklearn and its tools
from sklearn.metrics import accuracy_score

# changing data to tensors
x_train = torch.from_numpy(times_train).reshape(10,1)
x_val = torch.from_numpy(times_val).reshape(10,1)
target_train = torch.from_numpy(D_train).reshape(10,1)
target_val = torch.from_numpy(D_val).reshape(10,1)

# setting hyperparameters and gettings epoch sizes
batch_size = 10
num_epochs = 100
num_samples_train = len(x_train)
num_batches_train = num_samples_train // batch_size
num_samples_valid = len(target_val)
num_batches_valid = num_samples_valid // batch_size

# setting up lists for handling loss/accuracy
train_acc, train_loss = [], []
valid_acc, valid_loss = [], []
test_acc, test_loss = [], []
cur_loss = 0
losses = []

get_slice = lambda i, size: range(i * size, (i + 1) * size)

for epoch in range(num_epochs):
    # Forward -> Backprob -> Update params
    ## Train
    cur_loss = 0
    net.train()
    for i in range(num_batches_train):
        optimizer.zero_grad()
        slce = get_slice(i, batch_size)
        output = net(x_train[slce])
        
        # compute gradients given loss
        target_batch = target_train[slce]
        batch_loss = criterion(output, target_batch)
        batch_loss.backward()
        optimizer.step()
        
        cur_loss += batch_loss   
    losses.append(cur_loss / batch_size)

    net.eval()
    ### Evaluate training
    train_preds, train_targs = [], []
    for i in range(num_batches_train):
        slce = get_slice(i, batch_size)
        output = net(x_train[slce])

        preds = output
        # preds = torch.max(output, 1)[1]
        
        train_targs += list(target_train[slce].numpy())
        train_preds += list(preds.data.numpy())
        

    # train_acc_cur = accuracy_score(train_targs, train_preds)
    # train_acc_cur = 
    
    # train_acc.append(train_acc_cur)
    
    if epoch % 10 == 0:
        print("Epoch %2i : Train Loss %f" % (
                epoch+1, losses[-1]))

epoch = np.arange(epoch)
plt.figure()
plt.plot(epoch, train_acc, 'r', epoch)
plt.legend(['Train Accucary'])
plt.xlabel('Updates'), plt.ylabel('Acc')

Epoch  1 : Train Loss 0.021634


Epoch 11 : Train Loss 0.021582
Epoch 21 : Train Loss 0.021531
Epoch 31 : Train Loss 0.021481
Epoch 41 : Train Loss 0.021431
Epoch 51 : Train Loss 0.021383
Epoch 61 : Train Loss 0.021335
Epoch 71 : Train Loss 0.021287
Epoch 81 : Train Loss 0.021240
Epoch 91 : Train Loss 0.021194
